# Comment scraping

Collect comments from 56 video links across TikTok, X, Instagram, YouTube and Facebook.

CSV columns: **like_count**, **reply_count**, **date_published** (UTC). Unknown values stay missing. No account scraping, profile enrichment, behavioral features or model training.

YouTube uses `YOUTUBE_API_KEY` from `.env`. Browser platforms pause for login in a visible Chromium window. Run cells from top to bottom. Raw comment responses stay local for deduplication/replay; they can include inline author metadata. The exported CSV contains only the three columns.

Install: `pip install pandas python-dotenv google-api-python-client playwright` then `playwright install chromium`. Links are research candidates, not confirmed buzzer labels; some may be unavailable.


## 1. Environment

In [ ]:
import os, re, json, time
from pathlib import Path
from datetime import datetime, timezone
from typing import Iterable

import pandas as pd

COLLECTOR_VERSION = "local-e2e-1.1.0"

ROOT      = Path("./buzzer_data")
RAW       = ROOT / "raw"
CANONICAL = ROOT / "canonical"
FEATURES  = ROOT / "features"
PROFILE   = ROOT / "browser_profile"
for p in (RAW, CANONICAL, FEATURES, PROFILE):
    p.mkdir(parents=True, exist_ok=True)

from dotenv import load_dotenv
load_dotenv(override=False)
YOUTUBE_API_KEY = os.environ.get("YOUTUBE_API_KEY")

def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()

def jsonl_append(path: Path, rows: Iterable[dict]) -> int:
    n = 0
    with path.open("a", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False, default=str) + "\n"); n += 1
    return n

def jsonl_read(path: Path) -> list[dict]:
    if not path.exists(): return []
    with path.open(encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]

print("collector", COLLECTOR_VERSION)
print("YouTube key:", "set" if YOUTUBE_API_KEY else "not set; YouTube will be skipped")
def jsonl_iter(path: Path):
    """Stream payloads; report corrupt records instead of silently dropping observations."""
    if not path.exists():
        return
    with path.open(encoding="utf-8") as fh:
        for line_no, line in enumerate(fh, 1):
            if line.strip():
                try:
                    yield json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"{path}:{line_no}: invalid JSON") from exc


## 2. Video links

All 56 spreadsheet links are listed below. Change `MAX_COMMENTS` to adjust the per-video cap (includes replies).


In [ ]:
# Research candidates; buzzer activity is unverified.
MAX_COMMENTS = 500
VIDEO_URLS = {
    "tiktok": [
        "https://www.tiktok.com/@yehezsilva/video/7673383392859245831",
        "https://www.tiktok.com/@yanbo25/video/7538780346041552133",
        "https://www.tiktok.com/@ilhamber.2/video/7539883931223559432",
        "https://www.tiktok.com/@anies.hub/video/7416291932562853125",
        "https://www.tiktok.com/@muhammad.ilhamsyah23/video/7538464891959627024",
        "https://www.tiktok.com/@hamdiaadi/video/7673309762053770514",
        "https://www.tiktok.com/@garudasakti95/video/7544041220180430098",
        "https://www.tiktok.com/@memepolitaik1/video/7680473381287841045",
        "https://www.tiktok.com/@anomali.klipers/video/7675741914930433300",
        "https://www.tiktok.com/@lawliett41/video/7650964123953515792",
        "https://www.tiktok.com/@studio.musyafa/video/7650504991370693908",
        "https://www.tiktok.com/@cnnindonesia/video/7569811436185128213"
    ],
    "x": [
        "https://x.com/Gerindra/status/2052660507069563353",
        "https://x.com/Opposite6888/status/2050089921692741709",
        "https://x.com/menuembegejelek/status/2027687655543325106",
        "https://x.com/kumparan/status/1961964215759171756",
        "https://x.com/Muslim_AntiPKI9/status/2065705500432646610",
        "https://x.com/ommi_siregar/status/2097828809114599914",
        "https://x.com/prabowonomic/status/2089654506053390436",
        "https://x.com/BANGSAygSUJUD/status/2051496995626565887",
        "https://x.com/bangherwin/status/2003619134974886295",
        "https://x.com/ch_chotimah2/status/1973679381886451791",
        "https://x.com/CNNIndonesia/status/1970526403054706925"
    ],
    "instagram": [
        "https://www.instagram.com/p/Dc0WJjTlYbP/",
        "https://www.instagram.com/p/DdQ3mOXDatn/",
        "https://www.instagram.com/p/DccuOZZidPo/",
        "https://www.instagram.com/p/Dc-lkeqAD2V/",
        "https://www.instagram.com/p/Dc1FrjxPyrs/",
        "https://www.instagram.com/p/DciaPYlBM1U/",
        "https://www.instagram.com/p/Dc4xpy5pner/",
        "https://www.instagram.com/p/DcivgVVD8wh/",
        "https://www.instagram.com/p/DdEDFHRCM5w/",
        "https://www.instagram.com/p/Dcghyr1vwSp/",
        "https://www.instagram.com/p/DdD7OIwS8tG/"
    ],
    "youtube": [
        "https://www.youtube.com/watch?v=yHX7N-gcvhQ",
        "https://www.youtube.com/watch?v=wgpPtO2U844",
        "https://www.youtube.com/watch?v=MIo4tGN11j0",
        "https://www.youtube.com/watch?v=jyR79y-P1TI",
        "https://www.youtube.com/watch?v=iga-KngQGaU",
        "https://www.youtube.com/watch?v=-WLpXmnBmxo",
        "https://www.youtube.com/watch?v=dL8rbQ5xbdY",
        "https://www.youtube.com/watch?v=MkjOccts-K8",
        "https://www.youtube.com/watch?v=IQIS8TG5QlQ",
        "https://www.youtube.com/watch?v=QKt8Lp5j24Q",
        "https://www.youtube.com/watch?v=CXgWWyMYYb0",
        "https://www.youtube.com/watch?v=aCI89DbVW98"
    ],
    "facebook": [
        "https://www.facebook.com/beritaversiterbaik/videos/sri-mulyani-t4mpar-keras-dpr/1689639588386981/",
        "https://www.facebook.com/reel/1957079274999841/",
        "https://www.facebook.com/mdtelevisi/videos/video-joget-anggota-parlemen-yang-bikin-gempar/1206494331239538/",
        "https://www.facebook.com/KOMPAScom/videos/situasi-petamburan-memanas-saat-polisi-bubarkan-massa-usai-demo/1588750492103174/",
        "https://www.facebook.com/bpostonline/videos/dinonaktifkan-nasdem-dari-dpr-ri-ahmad-sahroni-minta-maaf-kepada-masyarakat-indo/1867642030459871/",
        "https://www.facebook.com/KantorBeritaPolitikRMOL/videos/mantan-panglima-tni-jenderal-purn-gatot-nurmantyo-mengisyaratkan-ada-gerakan-ter/1179987937483259/",
        "https://www.facebook.com/sripoku/videos/siapa-ali-ebrahimi-suami-salsa-erwina-hutagalung-sosok-pendukung-istri-yang-tant/1791460061453958/",
        "https://www.facebook.com/share/r/1CuBQabqBg/",
        "https://www.facebook.com/reel/1783402655901306",
        "https://www.facebook.com/reel/27239370689062407"
    ]
}

YT_VIDEO_IDS = [url.rsplit("v=", 1)[1] for url in VIDEO_URLS["youtube"]]
YT_MAX_PER_VIDEO = MAX_COMMENTS
BROWSER_TARGETS = [
    {"platform": platform, "url": url, "max_comments": MAX_COMMENTS}
    for platform, links in VIDEO_URLS.items() if platform != "youtube"
    for url in links
]
print(f"{len(YT_VIDEO_IDS)} YouTube videos; {len(BROWSER_TARGETS)} browser targets")


## 3. Shared comment helpers


In [ ]:
from urllib.parse import urlsplit, parse_qs

EXPORT_COLUMNS = ["like_count", "reply_count", "date_published"]
# IDs are used internally for deduplication and replies, never exported.
CANONICAL_FIELDS = ["post_id", "source_post_id", "parent_comment_id", "thread_id",
                    "created_at", "like_count", "reply_count"]
IDENTITY_FIELDS = {"post_id", "source_post_id", "parent_comment_id", "thread_id"}

def empty_record(**values):
    record = {field: values.get(field) for field in CANONICAL_FIELDS}
    for field in IDENTITY_FIELDS:
        record[field] = as_id(record[field])
    return record

def to_frame(records):
    return pd.DataFrame(records, columns=CANONICAL_FIELDS)

def write_csv(df, path):
    temporary = path.with_suffix(".tmp")
    df.to_csv(temporary, index=False, na_rep="\\N")
    temporary.replace(path)

def load_canonical(platform):
    path = CANONICAL / f"{platform}.csv"
    return pd.read_csv(path, keep_default_na=False, na_values=["\\N"]) if path.exists() else None

def scraping_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Export only requested comment metrics; unknown values remain null."""
    out = df.reindex(columns=["like_count", "reply_count"]).copy()
    dates = df["created_at"] if "created_at" in df else df.get("date_published", pd.Series(index=df.index, dtype="object"))
    out["date_published"] = pd.to_datetime(dates, utc=True, errors="coerce", format="mixed")
    return out

def save_canonical(df: pd.DataFrame, platform: str) -> Path:
    path = CANONICAL / f"{platform}.csv"
    write_csv(scraping_columns(df), path)
    return path

def as_id(value):
    if value is None or pd.isna(value): return None
    value = str(value).strip()
    return value if value and value not in {"0", "nan", "None", "<NA>"} else None

def source_post_id(platform, url):
    if not isinstance(url, str) or not url: return None
    parsed = urlsplit(url)
    query = parse_qs(parsed.query)
    if platform == "youtube":
        if query.get("v"): return query["v"][0]
        if parsed.hostname in {"youtu.be", "www.youtu.be"}: return parsed.path.strip("/") or None
    patterns = {"tiktok": r"/video/([^/?]+)", "instagram": r"/(?:p|reel|reels)/([^/?]+)",
                "x": r"/status/([^/?]+)", "facebook": r"/(?:posts|videos|reel)/(?:[^/]+/)?(\d+)(?:/|$)"}
    match = re.search(patterns.get(platform, r"/(?:shorts|embed)/([^/?]+)"), parsed.path)
    if match: return match.group(1)
    if platform == "facebook":
        for key in ("story_fbid", "fbid", "v"):
            if query.get(key): return query[key][0]
    return None

def timestamp_utc(value):
    if value is None: return None
    try:
        if isinstance(value, (int, float)) or (isinstance(value, str) and value.isdigit()):
            return datetime.fromtimestamp(float(value), timezone.utc).isoformat()
        ts = pd.to_datetime(value, utc=True, errors="coerce")
        return None if pd.isna(ts) else ts.isoformat()
    except (ValueError, TypeError, OverflowError, OSError):
        return None


## 4. YouTube — runs unattended

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

def yt_fetch(video_id, max_comments, yt=None):
    """Fetch top-level comments and paginated replies within one shared hard cap."""
    if max_comments < 0:
        raise ValueError('max_comments must be nonnegative')
    if max_comments == 0:
        return []
    yt = yt or build('youtube', 'v3', developerKey=YOUTUBE_API_KEY, cache_discovery=False)
    out, seen, token, tokens = ([], set(), None, set())

    def add(kind, item, parent=None):
        cid = item['snippet']['topLevelComment']['id'] if kind == 'top' else item['id']
        if cid not in seen and len(out) < max_comments:
            seen.add(cid)
            out.append({'_kind': kind, '_video_id': video_id, '_parent': parent, '_captured_at': now_utc(), '_max_comments': max_comments, 'item': item})
    while len(out) < max_comments:
        response = yt.commentThreads().list(part='snippet,replies', videoId=video_id, maxResults=min(100, max_comments - len(out)), pageToken=token, textFormat='plainText', order='time').execute()
        for item in response.get('items', []):
            add('top', item)
            parent = item['snippet']['topLevelComment']['id']
            embedded = item.get('replies', {}).get('comments', [])
            for reply in embedded:
                add('reply', reply, parent)
            if len(embedded) < item['snippet'].get('totalReplyCount', 0):
                reply_token, reply_tokens = (None, set())
                while len(out) < max_comments:
                    response_replies = yt.comments().list(part='snippet', parentId=parent, maxResults=min(100, max_comments - len(out)), pageToken=reply_token, textFormat='plainText').execute()
                    for reply in response_replies.get('items', []):
                        add('reply', reply, parent)
                    reply_token = response_replies.get('nextPageToken')
                    if not reply_token or reply_token in reply_tokens:
                        break
                    reply_tokens.add(reply_token)
            if len(out) >= max_comments:
                break
        token = response.get('nextPageToken')
        if not token or token in tokens:
            break
        tokens.add(token)
    return out

def yt_normalise(raw):
    top = raw['_kind'] == 'top'
    item = raw['item']
    comment = item['snippet']['topLevelComment'] if top else item
    sn = comment['snippet']
    parent = sn.get('parentId') or raw.get('_parent') if not top else None
    return empty_record(source_post_id=raw['_video_id'], post_id=comment['id'], parent_comment_id=parent, thread_id=parent or comment['id'], created_at=sn.get('publishedAt'), like_count=sn.get('likeCount'), reply_count=item['snippet'].get('totalReplyCount') if top else None)

def run_youtube_collection():
    if not YOUTUBE_API_KEY:
        print('YOUTUBE_API_KEY not set — skipping YouTube')
        return None
    if not YT_VIDEO_IDS:
        print('no YT_VIDEO_IDS configured — skipping YouTube')
        return None
    recs = []
    for vid in YT_VIDEO_IDS:
        try:
            raw = yt_fetch(vid, YT_MAX_PER_VIDEO)
        except HttpError as exc:
            # Applies to both comment-thread and reply-page requests.
            try:
                details = json.loads(exc.content).get('error', {}).get('errors', [])
                reasons = {item.get('reason') for item in details}
            except (ValueError, TypeError, AttributeError):
                reasons = set()
            if reasons and reasons <= {'videoNotFound', 'commentsDisabled'}:
                print(f'  {vid}: skipped ({", ".join(sorted(reasons))})')
                continue
            # HttpError includes the request URL/API key: never chain or print it.
            raise RuntimeError(
                f'YouTube collection failed for {vid} (HTTP {exc.resp.status}); '
                'check API access/quota. Previous CSV preserved; completed videos remain in raw JSONL.'
            ) from None
        jsonl_append(RAW / 'youtube_raw.jsonl', raw)
        recs += [yt_normalise(r) for r in raw]
        print(f'  {vid}: {len(raw)} comments')
    if not recs:
        print('youtube: nothing collected')
        return None
    df = to_frame(recs).drop_duplicates(subset=['post_id'], keep='last')
    save_canonical(df, 'youtube')
    print(f'youtube: {len(df)} comments')
    return df

df_yt = run_youtube_collection()


## 5. Browser platforms — TikTok, Instagram, X, Facebook — the part that pauses for you

`Harvester` opens a real, visible Chromium window and records the JSON that the comment section
actually loads over the network, rather than parsing the DOM. It's the SAME class and the same
scroll/batch/coverage machinery for all four platforms below — only the endpoint patterns,
comment-panel selector, and reply-button text differ per platform (see the dicts at the top of
the next cell). Each platform uses its own persistent profile
(`browser_profile/<platform>/`), so a login you complete once is still there next time you run
this notebook.

**When you run the next cell:** for each platform present in `BROWSER_TARGETS`, a browser opens.
Log in if asked. Then come back to the notebook and press **Enter in the input box that appears
below the cell** — execution resumes automatically and scrolls the comment section for you. This
is the one genuinely manual step in the whole notebook, and it is manual because making it
automatic would mean automating a login, which is where "scraping public data" turns into
"circumventing platform security controls."

A few things differ meaningfully between platforms, worth knowing before you collect:
- **TikTok and Instagram** render a post at desktop width with a dedicated comments panel, so
  the harvester hovers that panel before scrolling.
- **X** has no such panel — replies scroll with the whole page/timeline — and can interleave
  ads/promoted tweets while scrolling, which is why its example config in §2 uses a higher
  `idle_limit`.
- **Facebook** is the least stable of the four: its GraphQL doc_ids rotate constantly and the
  comment payload shape has changed more than once. Expect to lean on `inspect_payloads` (§6)
  more here than on the other platforms if a run captures 0 or very few comments.

In [ ]:
# Sync Playwright API, driven from a dedicated worker thread.
#
# Two separate Windows/Jupyter issues stack here, and both need fixing:
#
# 1. ipykernel keeps an asyncio loop running in the main thread for its own use (zmq comms),
#    and Playwright's sync API refuses to run inside a thread that already has a loop
#    running ("Sync API inside the asyncio loop" error). Fix: call it from a fresh thread
#    that has no loop of its own — the ThreadPoolExecutor below.
# 2. That's not sufficient on Windows. ipykernel deliberately sets the *global* asyncio
#    event loop policy to WindowsSelectorEventLoopPolicy at startup, because pyzmq (which
#    the kernel's comms depend on) doesn't support ProactorEventLoop. That policy is
#    process-wide, not thread-local, so even a brand-new thread inherits it — and
#    Playwright's driver process needs Proactor to be spawned at all, hence the
#    NotImplementedError. Fix: explicitly switch the policy to Proactor before Playwright
#    starts. This is safe: it only changes what future new_event_loop() calls hand out —
#    the kernel's own loop object was already created at startup and keeps running as-is,
#    unaffected by a later policy change.
import sys, asyncio
if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

from playwright.sync_api import sync_playwright
from concurrent.futures import ThreadPoolExecutor

# One Harvester drives all four platforms — only the bits below differ per platform. Endpoint
# substrings are how each platform's comment payload gets recognised on the wire; everything
# else (scrolling, batching, flush/cache) is shared in the class further down.
COMMENT_ENDPOINTS = {
    "tiktok":    ["/api/comment/list", "/api/comment/reply"],
    # IG's classic private-API path. IG has intermittently routed comment loading through
    # GraphQL (`/graphql/query` + a doc_id) instead on some rollouts — if a run captures 0
    # payloads, check the Network tab on a real post and add whatever URL actually fires.
    "instagram": ["/api/v1/media/", "/comments/"],
    # X's GraphQL operation name is baked into the URL path itself
    # (".../i/api/graphql/<hash>/TweetDetail"), so matching the operation name survives the
    # hash rotating on every X deploy, which matching the hash itself would not.
    "x":         ["TweetDetail", "ConversationTimeline"],
    # Facebook's GraphQL doc_ids rotate constantly and carry no stable name in the URL, so
    # this is the widest (least precise) pattern here — parse_facebook sifts payloads by
    # shape rather than trusting the URL alone. Expect this one to need the most hand-tuning.
    "facebook":  ["/api/graphql/"],
}

# Selector guesses for each platform's scrollable comment container, tried in order. TikTok and
# Instagram render a single post at desktop width as media-left / comments-right, so hovering
# the right-hand panel before wheeling is what makes page.mouse.wheel() land on the comment list
# instead of the whole document. X has no such panel — replies scroll with the main timeline —
# and Facebook's DOM is too unstable to target reliably, so both just use the viewport fallback.
COMMENT_PANEL_SELECTORS = {
    "tiktok":    ['[data-e2e="comment-list"]'],
    "instagram": ['section main ul', 'article ul'],
}
PANEL_FALLBACK = {  # (x_fraction, y_fraction) of the viewport, used when no selector matches
    "tiktok": (0.75, 0.5), "instagram": (0.75, 0.5), "x": (0.5, 0.5), "facebook": (0.5, 0.5),
}

# "view replies" button text varies per platform (and locale — id-ID strings included since
# the persistent context below runs in id-ID).
REPLY_BUTTON_PATTERNS = {
    "tiktok":    r"repl(y|ies)|balasan",
    "instagram": r"view (\d+ )?repl(y|ies)|lihat balasan",
    "x":         r"show (more )?repl(y|ies)|more repl(y|ies)",
    "facebook":  r"view (more|\d+) (repl(y|ies)|comments?)|lihat balasan|balasan lainnya",
}

# Populated by run_browser_collection: platform -> unique comments captured THIS session
# (before raw JSONL accumulation from earlier runs). §6 uses this to explain any gap between
# "what this run scrolled" and "what's in the canonical file" (which includes older runs too).
SESSION_SEEN: dict[str, int] = {}

# Coverage counts are diagnostic candidates. Platform totals can count different populations;
# the collector stops on a cap/plateau/batch ceiling, never on an unverified percentage.
def _walk_for_ids_and_totals(node):
    """Count comment-shaped nodes only; profile/post engagement totals are not coverage."""
    stack = [node]
    while stack:
        item = stack.pop()
        if isinstance(item, dict):
            cid = None
            if "cid" in item and "text" in item: cid = item["cid"]
            elif "text" in item and isinstance(item.get("user"), dict):
                cid = item.get("pk") or item.get("id")
            elif isinstance(item.get("legacy"), dict) and "full_text" in item["legacy"]:
                legacy = item["legacy"]
                if legacy.get("in_reply_to_status_id_str"):
                    cid = item.get("rest_id") or legacy.get("id_str")
                if cid is not None: yield "id", str(cid)
                continue  # quoted tweets and their users are not additional comments
            elif item.get("__typename") == "Comment" or "parent_comment" in item:
                cid = item.get("id") or item.get("legacy_fbid")
            if cid is not None: yield "id", str(cid)
            stack.extend(item.values())
        elif isinstance(item, list): stack.extend(item)

def coverage_parts(row):
    if row.get("_kind", "comment") != "comment": return set(), None
    ids = {val for kind, val in _walk_for_ids_and_totals(row.get("payload")) if kind == "id"}
    if row.get("_platform") == "x":
        target = source_post_id("x", row.get("_page_url"))
        ids = set()
        stack = [row.get("payload")]
        while stack:
            node = stack.pop()
            if isinstance(node, dict):
                legacy = node.get("legacy")
                if isinstance(legacy, dict) and "full_text" in legacy:
                    tid = as_id(node.get("rest_id") or legacy.get("id_str"))
                    if tid and tid != target and target and (legacy.get("conversation_id_str") == target
                            or legacy.get("in_reply_to_status_id_str") == target): ids.add(tid)
                    continue
                stack.extend(node.values())
            elif isinstance(node, list): stack.extend(node)
    payload = row.get("payload", {})
    total = None
    # Only top-level comment lists have a comparable denominator. Reply totals are per thread.
    url = row.get("_url", "")
    if isinstance(payload, dict) and "comments" in payload and "reply" not in url and "child" not in url:
        for key in ("total", "comment_count", "total_comment_count"):
            value = payload.get(key)
            if isinstance(value, int) and not isinstance(value, bool) and value >= 0:
                total = value; break
    return ids, total

def estimate_coverage(rows):
    by_target = {}
    for row in rows:
        key = (row.get("_platform"), source_post_id(row.get("_platform"), row.get("_page_url")))
        ids, total = coverage_parts(row)
        state = by_target.setdefault(key, [set(), None])
        state[0].update(ids)
        if total is not None: state[1] = max(state[1] or 0, total)
    seen = sum(len(state[0]) for state in by_target.values())
    totals = [state[1] for state in by_target.values()]
    return seen, sum(totals) if totals and all(t is not None for t in totals) else None


class Harvester:
    def __init__(self, platform: str):
        self.platform = platform
        self.comment_patterns = COMMENT_ENDPOINTS[platform]
        self.patterns = self.comment_patterns
        self.captured, self.ctx, self.pw, self.page = [], None, None, None
        self._coverage_cursor, self._seen_ids, self._reported_total = 0, set(), None
        self.target_url, self.max_comments, self.capture_session = None, None, None

    def start(self):
        self.pw  = sync_playwright().start()
        self.ctx = self.pw.chromium.launch_persistent_context(
            user_data_dir=str(PROFILE / self.platform), headless=False,
            viewport={"width": 1440, "height": 900},
            locale="id-ID", timezone_id="Asia/Jakarta",
        )
        self.page = self.ctx.pages[0] if self.ctx.pages else self.ctx.new_page()
        self.page.on("response", self._on_response)
        return self

    def _on_response(self, resp):
        if not any(p in resp.url for p in self.patterns): return
        if getattr(resp, "status", 200) >= 400: return
        ct = (resp.headers.get("content-type") or "")
        if "json" not in ct and "text" not in ct: return
        try: body = resp.text()
        except Exception: return
        # Most platforms return one JSON object per response — try that first, the common
        # case. Facebook (and some X GraphQL batch calls) can return several newline-delimited
        # JSON objects in one body, each optionally prefixed with the "for (;;);"
        # anti-JSON-hijacking header Facebook has used for over a decade; fall back to
        # per-line parsing only if the whole-body parse fails.
        payloads = []
        try:
            payloads.append(json.loads(body))
        except json.JSONDecodeError:
            for line in body.split("\n"):
                line = line.strip()
                if not line: continue
                if line.startswith("for (;;);"):
                    line = line[len("for (;;);"):]
                try: payloads.append(json.loads(line))
                except json.JSONDecodeError: continue
        for payload in payloads:
            self.captured.append({"_platform": self.platform, "_url": resp.url,
                "_page_url": self.target_url or self.page.url, "_captured_at": now_utc(), "_kind": "comment",
                "_capture_session": self.capture_session, "_max_comments": self.max_comments,
                "payload": payload})

    def open(self, url: str, max_comments=None):
        self.target_url, self.max_comments, self.capture_session = url, max_comments, now_utc()
        self._coverage_cursor, self._seen_ids, self._reported_total = len(self.captured), set(), None
        self.page.goto(url, wait_until="domcontentloaded", timeout=60_000)
        self.page.wait_for_timeout(3000)

    def _hover_comment_panel(self):
        """
        page.mouse.wheel() fires at the CURRENT mouse position, which defaults to (0, 0) —
        the page's top-left corner — until something moves it. On platforms with a dedicated
        side comment panel (TikTok, Instagram), a wheel event at (0, 0) mostly misses it
        entirely; hovering the panel first fixes that. Platforms with no such panel (X, and
        Facebook by default) just use the viewport-fraction fallback.
        """
        for sel in COMMENT_PANEL_SELECTORS.get(self.platform, []):
            try:
                panel = self.page.locator(sel).first
                panel.wait_for(state="attached", timeout=3000)
                box = panel.bounding_box()
                if box:
                    self.page.mouse.move(box["x"] + box["width"] / 2, box["y"] + box["height"] / 2)
                    return
            except Exception:
                continue
        vp = self.page.viewport_size or {"width": 1440, "height": 900}
        fx, fy = PANEL_FALLBACK.get(self.platform, (0.5, 0.5))
        self.page.mouse.move(vp["width"] * fx, vp["height"] * fy)

    def _wait_for_growth(self, timeout_ms: int, poll_ms: int = 150):
        """
        Waits up to timeout_ms for a new payload to arrive, but returns the moment one does
        instead of always sleeping the full window. Genuinely idle rounds (nothing new
        loading) still pay the full timeout_ms — that's necessary to distinguish "slow" from
        "actually plateaued".
        """
        start = len(self.captured)
        elapsed = 0
        while elapsed < timeout_ms:
            self.page.wait_for_timeout(poll_ms)
            elapsed += poll_ms
            if len(self.captured) > start:
                return True
        return False

    def scroll_comments(self, max_rounds: int = 200, idle_limit: int = 8, pause_ms: int = 1800):
        """
        Scrolls until capture growth stalls for `idle_limit` consecutive rounds, or
        `max_rounds` is hit — not a fixed count. Lazy-loading batch size varies per platform
        (TikTok ~20/XHR, others similar order of magnitude), so this is driven by growth, not
        a fixed comment-per-round assumption.
        """
        self._hover_comment_panel()
        stagnant, last = 0, self._coverage_snapshot()[0]
        for i in range(max_rounds):
            if self._at_cap(): break
            self.page.mouse.wheel(0, 3200)
            self._wait_for_growth(pause_ms)
            seen = self._coverage_snapshot()[0]
            if seen == last:
                stagnant += 1
                if stagnant >= idle_limit:
                    print(f"    plateaued after round {i} ({len(self.captured)} payloads captured)")
                    break
            else:
                stagnant, last = 0, self._coverage_snapshot()[0]
            if (i + 1) % 10 == 0:
                print(f"    round {i+1}/{max_rounds}: {len(self.captured)} payloads captured")
        return len(self.captured)

    def expand_replies(self, max_clicks: int = 300, pause_ms: int = 900):
        """
        Clicks every visible 'view replies'-style button so nested replies load through the
        same endpoints the harvester already listens for. Best-effort: button text/selectors
        vary by platform and locale, so REPLY_BUTTON_PATTERNS may need adjusting — especially
        for Facebook, the least stable of the four. Each click waits only until its reply
        payload actually arrives (see _wait_for_growth).
        """
        pattern = REPLY_BUTTON_PATTERNS.get(self.platform)
        if not pattern:
            return 0
        clicked = 0
        for _ in range(max_clicks):
            if self._at_cap(): break
            buttons = self.page.locator(f"text=/{pattern}/i")
            if buttons.count() == 0:
                break
            try:
                buttons.first.click(timeout=3000)
                clicked += 1
                self._wait_for_growth(pause_ms)
            except Exception:
                break
        if clicked:
            print(f"    expanded {clicked} reply thread(s)")
        return clicked

    def _coverage_snapshot(self):
        """Incremental unique comment candidates and an optional reported-total hint."""
        # Each newly captured payload is scanned once, not again on every scroll/batch.
        while self._coverage_cursor < len(self.captured):
            row = self.captured[self._coverage_cursor]
            self._coverage_cursor += 1
            ids, total = coverage_parts(row)
            self._seen_ids.update(ids)
            if total is not None: self._reported_total = max(self._reported_total or 0, total)
        return len(self._seen_ids), self._reported_total

    def _at_cap(self):
        return self.max_comments is not None and self._coverage_snapshot()[0] >= self.max_comments

    def scroll_until_coverage(self, target_pct: float = 0.95, batch_rounds: int = 60,
                               max_batches: int = 8, idle_limit: int = 12,
                               pause_ms: int = 1800, expand_replies: bool = True,
                               max_comments: int | None = 500):
        """Stop on a per-target cap, two stagnant batches, or the batch ceiling.

        target_pct is a compatibility argument only: reported totals are not a verified
        denominator for captured comments/replies. Canonical replay enforces the hard cap
        on normalized records, because an individual network response can overshoot it.
        """
        if max_comments is not None and max_comments < 0: raise ValueError("negative comment cap")
        self.max_comments = max_comments
        prev_seen, idle_batches = self._coverage_snapshot()[0], 0
        stop_reason = "max_batches exhausted"
        for batch in range(max_batches):
            self.scroll_comments(max_rounds=batch_rounds, idle_limit=idle_limit, pause_ms=pause_ms)
            if expand_replies:
                self.expand_replies(pause_ms=max(pause_ms // 2, 600))
                self.scroll_comments(max_rounds=30, idle_limit=6, pause_ms=pause_ms)
            seen, total = self._coverage_snapshot()
            pct_str = f"reported total hint: {total}" if total is not None else "total unknown"
            print(f"    batch {batch+1}/{max_batches}: {seen} comments ({pct_str})"
                  + (f" — cap {max_comments}" if max_comments else ""))
            if max_comments is not None and seen >= max_comments:
                stop_reason = f"reached max_comments cap ({max_comments})"
                break
            # Report totals as hints only: top-level, reply and filtered totals are not
            # guaranteed to have the same denominator as the captured unique-comment count.
            idle_batches = idle_batches + 1 if seen == prev_seen else 0
            if idle_batches >= 2:
                stop_reason = "plateaued — two batches in a row with no new unique comments"
                break
            prev_seen = seen
        seen, total = self._coverage_snapshot()
        self.stop_reason = stop_reason
        print(f"    stopped: {stop_reason}")
        return seen, total

    def flush(self) -> int:
        n = jsonl_append(RAW / f"{self.platform}_payloads.jsonl", self.captured)
        self.captured = []
        self._coverage_cursor = 0
        return n

    def close(self):
        if self.ctx: self.ctx.close()
        if self.pw:  self.pw.stop()


def run_browser_collection(targets: list[dict]):
    """
    Groups targets by platform so login happens at most once per platform per run.
    Every Harvester call is submitted to the same single-worker thread and .result()
    is awaited immediately, so from the notebook's point of view this still runs top
    to bottom in order — the threading is invisible except that it fixes the Windows
    subprocess issue. input() prompts happen on the main thread as normal and block
    exactly as long as it takes you to log in and get ready; that's expected, not a hang.
    Each platform in `targets` gets its own persistent browser_profile/<platform>/ and its
    own login pause the first time — TikTok, Instagram, X and Facebook logins are unrelated.
    """
    if not targets:
        print("no BROWSER_TARGETS configured — skipping"); return

    by_platform: dict[str, list[dict]] = {}
    for t in targets:
        by_platform.setdefault(t["platform"], []).append(t)

    with ThreadPoolExecutor(max_workers=1) as ex:
        for platform, items in by_platform.items():
            print(f"\n=== {platform} ({len(items)} target(s)) ===")
            h = ex.submit(lambda p=platform: Harvester(p).start()).result()
            session_ids = set()
            try:
                for i, t in enumerate(items):
                    ex.submit(h.open, t["url"], t.get("max_comments", 500)).result()
                    if i == 0:
                        input(f"  [{platform}] Log in if prompted, then press Enter here to continue... ")
                    print(f"  scrolling: {t['url']}")
                    seen, total = ex.submit(h.scroll_until_coverage,
                        t.get("target_coverage", 0.95), t.get("batch_rounds", 60),
                        t.get("max_batches", 8), t.get("idle_limit", 12),
                        t.get("pause_ms", 1800), t.get("expand_replies", True),
                        t.get("max_comments", 500)).result()
                    session_ids.update(h._seen_ids)
                    jsonl_append(RAW / "collection_runs.jsonl", [{
                        "platform": platform, "source_post_id": source_post_id(platform, t["url"]),
                        "capture_session": h.capture_session, "captured_unique": seen, "stop_reason": h.stop_reason,
                        "reported_total": total, "max_comments": t.get("max_comments", 500),
                        "collected_at": now_utc(), "sampling_method": "browser_display_order",
                    }])
                    ex.submit(h.flush).result()
                SESSION_SEEN[platform] = len(session_ids)
            finally:
                try: ex.submit(h.flush).result()
                finally: ex.submit(h.close).result()

run_browser_collection(BROWSER_TARGETS)


## 6. Parse the captured browser payloads

In [ ]:
def dig(obj, *paths, default=None):
    for path in paths:
        cur, ok = (obj, True)
        for key in path.split('.'):
            if isinstance(cur, dict) and key in cur:
                cur = cur[key]
            elif isinstance(cur, list) and key.isdigit() and (int(key) < len(cur)):
                cur = cur[int(key)]
            else:
                ok = False
                break
        if ok and cur is not None:
            return cur
    return default

def parse_tiktok(row):
    out = []
    stack = [(c, None) for c in reversed(dig(row['payload'], 'comments', default=[]) or [])]
    seen = set()
    while stack:
        c, enclosing_parent = stack.pop()
        cid = as_id(c.get('cid'))
        if not cid or cid in seen:
            continue
        seen.add(cid)
        stack.extend(((child, cid) for child in reversed(c.get('reply_comment') or [])))
        reply_id = as_id(c.get('reply_id')) or enclosing_parent
        is_reply = bool(reply_id and reply_id != '0')
        thread_id = reply_id if is_reply else c.get('cid')
        out.append(empty_record(post_id=c.get('cid'), thread_id=thread_id, source_post_id=c.get('aweme_id') or source_post_id('tiktok', row.get('_page_url')), parent_comment_id=as_id(c.get('reply_to_reply_id')) or reply_id, created_at=timestamp_utc(c.get('create_time')), like_count=c.get('digg_count'), reply_count=c.get('reply_comment_total')))
    return out

def parse_instagram(row):
    """
    Instagram's private-API comments endpoint (`/api/v1/media/<id>/comments/`) — the same shape
    tools like instaloader/instagrapi have relied on for years. Handles both the top-level
    "comments" list and the "child_comments" list returned when a reply thread is expanded.
    """
    out = []
    payload = row['payload']
    endpoint_parent = re.search('/comments/([^/]+)/', row.get('_url', ''))
    endpoint_parent = endpoint_parent.group(1) if endpoint_parent else None
    items = [(c, None) for c in payload.get('comments', []) or []]
    items += [(c, endpoint_parent) for c in payload.get('child_comments', []) or []]
    seen = set()
    for c, enclosing_parent in items:
        cid = as_id(c.get('pk') or c.get('id'))
        if not cid or cid in seen:
            continue
        seen.add(cid)
        items.extend(((child, cid) for child in c.get('preview_child_comments', []) or []))
        items.extend(((child, cid) for child in c.get('child_comments', []) or []))
        parent_id = as_id(c.get('parent_comment_id')) or enclosing_parent
        is_reply = bool(parent_id)
        cid = c.get('pk') or c.get('id')
        thread_id = str(parent_id) if is_reply else str(cid or '')
        out.append(empty_record(post_id=cid, thread_id=thread_id, parent_comment_id=parent_id, created_at=timestamp_utc(c.get('created_at')), like_count=c.get('comment_like_count'), reply_count=c.get('child_comment_count')))
    return out

def _parse_x_timestamp(s):
    if not s:
        return None
    try:
        return datetime.strptime(s, '%a %b %d %H:%M:%S %z %Y').isoformat()
    except (ValueError, TypeError):
        return None

def _iter_x_tweet_nodes(node):
    """
    Recursively finds every dict shaped like an X/Twitter GraphQL Tweet result, regardless of
    how deep the surrounding instructions/entries/itemContent wrapper nests it — that wrapper
    has changed shape before (TimelineAddEntries vs TimelineAddEntry, threaded conversations vs
    single-tweet detail) and searching structurally survives that better than hardcoding a path.
    """
    if isinstance(node, dict):
        legacy = node.get('legacy')
        if isinstance(legacy, dict) and 'full_text' in legacy:
            yield node
            return
        elif isinstance(node.get('tweet'), dict) and isinstance(node['tweet'].get('legacy'), dict):
            yield node['tweet']
            return
        for v in node.values():
            yield from _iter_x_tweet_nodes(v)
    elif isinstance(node, list):
        for v in node:
            yield from _iter_x_tweet_nodes(v)

def parse_x(row):
    """Extract replies to the selected X post."""
    out = []
    for t in _iter_x_tweet_nodes(row['payload']):
        legacy = t.get('legacy', {}) or {}
        tid = t.get('rest_id') or legacy.get('id_str')
        conv_id = legacy.get('conversation_id_str')
        target_id = source_post_id('x', row.get('_page_url'))
        if not target_id or str(tid) == target_id:
            continue
        if str(conv_id) != target_id and str(legacy.get('in_reply_to_status_id_str')) != target_id:
            continue
        is_reply = bool(legacy.get('in_reply_to_status_id_str'))
        thread_id = conv_id or tid
        out.append(empty_record(post_id=tid, thread_id=thread_id, source_post_id=target_id, parent_comment_id=legacy.get('in_reply_to_status_id_str'), created_at=_parse_x_timestamp(legacy.get('created_at')), like_count=legacy.get('favorite_count'), reply_count=legacy.get('reply_count')))
    return out

def _iter_facebook_comment_nodes(node):
    """
    Facebook's GraphQL comment shape is the least stable of the four platforms here — doc_ids
    rotate constantly and Meta has restructured the UFI/comment payload multiple times. This
    searches structurally for any dict carrying a text body ('body'/'message', each usually
    {'text': ...}) alongside an 'author', rather than trusting one fixed path. Confirm this
    actually finds your comments with inspect_payloads('facebook') before trusting a run's
    output, and adjust the field names below to match what you see if it comes back empty.
    """
    if isinstance(node, dict):
        body = node.get('body') or node.get('message')
        text = body.get('text') if isinstance(body, dict) else None
        if text is not None and isinstance(node.get('author'), dict) and (node.get('__typename') == 'Comment' or 'parent_comment' in node):
            yield node
        for v in node.values():
            yield from _iter_facebook_comment_nodes(v)
    elif isinstance(node, list):
        for v in node:
            yield from _iter_facebook_comment_nodes(v)

def parse_facebook(row):
    out = []
    for c in _iter_facebook_comment_nodes(row['payload']):
        cid = c.get('id') or c.get('legacy_fbid')
        parent = dig(c, 'parent_comment.id', default=None)
        is_reply = bool(parent)
        thread_id = str(parent) if is_reply else str(cid or '')
        created = c.get('created_time')
        out.append(empty_record(post_id=cid, thread_id=thread_id, parent_comment_id=parent, created_at=timestamp_utc(created), like_count=dig(c, 'feedback.like_count.count', 'feedback.like_count', default=None), reply_count=dig(c, 'feedback.replies_fields.total_count', default=None)))
    return out

def inspect_payloads(platform: str, n: int=2):
    for row in jsonl_read(RAW / f'{platform}_payloads.jsonl')[:n]:
        print('URL:', row['_url'][:110])
        print(json.dumps(row['payload'], ensure_ascii=False)[:1200], '\n---')

def build_browser_canonical(platform: str):
    recs, errors, payload_count = ({}, 0, 0)
    session_ids = {}
    for row in jsonl_iter(RAW / f'{platform}_payloads.jsonl'):
        if row.get('_kind', 'comment') != 'comment':
            continue
        payload_count += 1
        try:
            for rec in PARSERS[platform](row):
                if rec.get('post_id'):
                    rec.update(_capture_session=row.get('_capture_session'), _sampling_method='browser_display_order', _requested_comment_cap=row.get('_max_comments'))
                    key = (row.get('_capture_session'), row.get('_page_url'))
                    seen = session_ids.setdefault(key, set())
                    cap = row.get('_max_comments')
                    if cap is not None and len(seen) >= cap and (rec['post_id'] not in seen):
                        continue
                    seen.add(rec['post_id'])
                    recs[rec['post_id']] = rec
        except (TypeError, ValueError, KeyError, AttributeError) as exc:
            errors += 1
            print(f'  ! parse error in payload {payload_count}: {type(exc).__name__}: {exc}')
    if errors:
        raise ValueError(f'{platform}: {errors} payloads failed; previous canonical preserved')
    if not recs:
        if payload_count:
            raise ValueError(f'{platform}: payloads captured but zero comments parsed')
        print(f'{platform}: no payloads captured')
        return None
    df = to_frame(list(recs.values()))
    save_canonical(df, platform)
    print(f'{platform}: {len(df)} unique archived comments')
    return df

PARSERS = {"tiktok": parse_tiktok, "instagram": parse_instagram, "x": parse_x, "facebook": parse_facebook}

results_browser = {p: build_browser_canonical(p) for p in sorted({t["platform"] for t in BROWSER_TARGETS})}


## 7. Export comments

Per-platform CSVs: `buzzer_data/canonical/<platform>.csv`. Combined CSV: `buzzer_data/features/dataset.csv`. Browser replay includes earlier locally archived comments. Old account exports are not deleted or regenerated.


In [ ]:
def assemble_scraped_data(platforms=("youtube", "tiktok", "instagram", "x", "facebook")):
    """Combine available platform CSVs in the same three-column format."""
    frames = [scraping_columns(load_canonical(p)) for p in platforms
              if (CANONICAL / f"{p}.csv").exists()]
    if not frames:
        raise FileNotFoundError("no canonical CSV files - nothing was collected")
    out = pd.concat(frames, ignore_index=True)
    write_csv(out, FEATURES / "dataset.csv")
    return out


df = assemble_scraped_data()
print(f"Exported {len(df)} comments with columns: {list(df.columns)}")
df.head()


## Notes

- Browser endpoints can change; inspect captured payloads if collection fails. Login or private/unavailable content can prevent collection.
- Missing counts remain missing; a Facebook reaction total is not substituted for likes.
- Comment IDs are used internally to deduplicate in linear time; identical metric values are not duplicates.
- These three fields describe engagement and timing; they do not establish comment bias or buzzer activity.
- Offline checks: `python -m unittest discover -s tests -v`.
